# PAATRA Step 1: Parameter Allocation Audit

Load small language models and break down where every parameter lives:  
**embeddings (vocabulary)** vs **transformer blocks (reasoning)**.

This validates the core PAATRA observation: small models inherited from big families waste a huge fraction of their parameters on oversized vocabulary tables.

In [1]:
!pip install -q torch transformers accelerate

In [2]:
import torch
from transformers import AutoModelForCausalLM
import gc

def audit_model(model_id: str):
    """Load a model and break down parameter allocation."""
    print(f"\n{'='*60}")
    print(f"  {model_id}")
    print(f"{'='*60}")

    print("Loading model...")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float32,
        device_map="cpu",
    )
    config = model.config

    vocab_size = config.vocab_size
    hidden_dim = config.hidden_size
    num_layers = config.num_hidden_layers
    print(f"\nConfig: vocab={vocab_size:,}  hidden={hidden_dim}  layers={num_layers}")

    # Bucket every parameter
    embedding_params = 0
    transformer_params = 0
    lm_head_params = 0
    other_params = 0

    for name, param in model.named_parameters():
        count = param.numel()
        if "embed_tokens" in name or "wte" in name:
            embedding_params += count
        elif "lm_head" in name:
            lm_head_params += count
        elif any(k in name for k in ["layers.", "block.", "h."]):
            transformer_params += count
        else:
            other_params += count

    # Check if lm_head is tied (shares weights with embed_tokens)
    tied = False
    if hasattr(model, "lm_head") and hasattr(model, "get_input_embeddings"):
        emb_weight = model.get_input_embeddings().weight
        head_weight = model.lm_head.weight
        if emb_weight.data_ptr() == head_weight.data_ptr():
            tied = True

    total = sum(p.numel() for p in model.parameters())
    vocab_total = embedding_params
    if not tied:
        vocab_total += lm_head_params

    print(f"\n--- Parameter Breakdown ---")
    print(f"  Input embeddings:   {embedding_params:>12,}  ({embedding_params/total*100:5.1f}%)")
    if tied:
        print(f"  LM head (output):   {'[tied to input]':>12}")
    else:
        print(f"  LM head (output):   {lm_head_params:>12,}  ({lm_head_params/total*100:5.1f}%)")
    print(f"  Transformer blocks: {transformer_params:>12,}  ({transformer_params/total*100:5.1f}%)")
    print(f"  Other (norms etc):  {other_params:>12,}  ({other_params/total*100:5.1f}%)")
    print(f"  {'─'*42}")
    print(f"  TOTAL:              {total:>12,}")

    print(f"\n--- The PAATRA Question ---")
    print(f"  Vocabulary cost:    {vocab_total:,} params = {vocab_total/total*100:.1f}% of model")
    print(f"  Reasoning capacity: {transformer_params:,} params = {transformer_params/total*100:.1f}% of model")
    ratio = vocab_total / transformer_params if transformer_params > 0 else float('inf')
    print(f"  Vocab-to-Transformer ratio: {ratio:.2f}x")

    print(f"\n--- What-If: Reallocate to 10K vocab ---")
    small_vocab_params = 10_000 * hidden_dim
    freed = vocab_total - small_vocab_params
    new_transformer = transformer_params + freed
    print(f"  10K vocab embedding: {small_vocab_params:,} params")
    print(f"  Freed params:        {freed:,}")
    print(f"  New transformer capacity: {new_transformer:,} ({new_transformer/transformer_params:.1f}x current)")

    result = {
        "model": model_id,
        "total": total,
        "vocab": vocab_total,
        "transformer": transformer_params,
        "vocab_pct": vocab_total / total * 100,
        "transformer_pct": transformer_params / total * 100,
        "vocab_size": vocab_size,
        "hidden_dim": hidden_dim,
        "num_layers": num_layers,
        "tied": tied,
    }

    del model
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return result

## Config-Only Audit (Gated Models)

These models need HuggingFace auth, so we compute from known architecture specs.  
If you have access, you can move them to the real audit section below.

In [3]:
print(f"{'='*60}")
print(f"  Config-Only Audit (gated models)")
print(f"{'='*60}")

# (name, vocab_size, hidden_dim, num_layers, tied_embeddings)
gated_models = [
    ("Gemma 3 270M",  262_144, 1536, 18, True),
    ("Llama 3.2 1B",  128_256, 2048, 16, False),
    ("Llama 3.2 3B",  128_256, 3072, 28, False),
    ("Gemma 2 2B",    256_000, 2304, 26, True),
]

for name, vocab, hidden, layers, tied in gated_models:
    emb = vocab * hidden
    # Rough: 4*h^2 (attn) + 8*h^2 (FFN) = 12*h^2 per layer
    trans_est = layers * 12 * hidden * hidden
    total_est = emb + trans_est + (0 if tied else emb)
    vocab_cost = emb
    if not tied:
        vocab_cost = emb * 2  # input + output

    print(f"\n  {name}")
    print(f"    Vocab={vocab:,}  Hidden={hidden}  Layers={layers}  Tied={'yes' if tied else 'no'}")
    print(f"    Embedding params:    {emb:>12,}  ({emb/total_est*100:.1f}%)")
    print(f"    Transformer (est):   {trans_est:>12,}  ({trans_est/total_est*100:.1f}%)")
    print(f"    Total (est):         {total_est:>12,}")
    print(f"    Vocab cost:          {vocab_cost/total_est*100:.1f}% of model")

  Config-Only Audit (gated models)

  Gemma 3 270M
    Vocab=262,144  Hidden=1536  Layers=18  Tied=yes
    Embedding params:     402,653,184  (44.1%)
    Transformer (est):    509,607,936  (55.9%)
    Total (est):          912,261,120
    Vocab cost:          44.1% of model

  Llama 3.2 1B
    Vocab=128,256  Hidden=2048  Layers=16  Tied=no
    Embedding params:     262,668,288  (19.7%)
    Transformer (est):    805,306,368  (60.5%)
    Total (est):         1,330,642,944
    Vocab cost:          39.5% of model

  Llama 3.2 3B
    Vocab=128,256  Hidden=3072  Layers=28  Tied=no
    Embedding params:     394,002,432  (10.0%)
    Transformer (est):   3,170,893,824  (80.1%)
    Total (est):         3,958,898,688
    Vocab cost:          19.9% of model

  Gemma 2 2B
    Vocab=256,000  Hidden=2304  Layers=26  Tied=yes
    Embedding params:     589,824,000  (26.3%)
    Transformer (est):   1,656,225,792  (73.7%)
    Total (est):         2,246,049,792
    Vocab cost:          26.3% of model


## Real Model Audit (Ungated Models)

These download and inspect actual weights. No auth needed.

In [4]:
models_to_audit = [
    "Qwen/Qwen2.5-0.5B",                      # 151K vocab, severe overhead
    "HuggingFaceTB/SmolLM2-360M",              # 49K vocab, better ratio
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",      # 32K vocab, good baseline
]

results = []
for m in models_to_audit:
    try:
        results.append(audit_model(m))
    except Exception as e:
        print(f"\nFailed to load {m}: {e}")


  Qwen/Qwen2.5-0.5B
Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


`torch_dtype` is deprecated! Use `dtype` instead!



Config: vocab=151,936  hidden=896  layers=24

--- Parameter Breakdown ---
  Input embeddings:    136,134,656  ( 27.6%)
  LM head (output):   [tied to input]
  Transformer blocks:  357,897,216  ( 72.4%)
  Other (norms etc):           896  (  0.0%)
  ──────────────────────────────────────────
  TOTAL:               494,032,768

--- The PAATRA Question ---
  Vocabulary cost:    136,134,656 params = 27.6% of model
  Reasoning capacity: 357,897,216 params = 72.4% of model
  Vocab-to-Transformer ratio: 0.38x

--- What-If: Reallocate to 10K vocab ---
  10K vocab embedding: 8,960,000 params
  Freed params:        127,174,656
  New transformer capacity: 485,071,872 (1.4x current)

  HuggingFaceTB/SmolLM2-360M
Loading model...



Config: vocab=49,152  hidden=960  layers=32

--- Parameter Breakdown ---
  Input embeddings:     47,185,920  ( 13.0%)
  LM head (output):   [tied to input]
  Transformer blocks:  314,634,240  ( 87.0%)
  Other (norms etc):           960  (  0.0%)
  ──────────────────────────────────────────
  TOTAL:               361,821,120

--- The PAATRA Question ---
  Vocabulary cost:    47,185,920 params = 13.0% of model
  Reasoning capacity: 314,634,240 params = 87.0% of model
  Vocab-to-Transformer ratio: 0.15x

--- What-If: Reallocate to 10K vocab ---
  10K vocab embedding: 9,600,000 params
  Freed params:        37,585,920
  New transformer capacity: 352,220,160 (1.1x current)

  TinyLlama/TinyLlama-1.1B-Chat-v1.0
Loading model...



Config: vocab=32,000  hidden=2048  layers=22

--- Parameter Breakdown ---
  Input embeddings:     65,536,000  (  6.0%)
  LM head (output):     65,536,000  (  6.0%)
  Transformer blocks:  968,974,336  ( 88.1%)
  Other (norms etc):         2,048  (  0.0%)
  ──────────────────────────────────────────
  TOTAL:              1,100,048,384

--- The PAATRA Question ---
  Vocabulary cost:    131,072,000 params = 11.9% of model
  Reasoning capacity: 968,974,336 params = 88.1% of model
  Vocab-to-Transformer ratio: 0.14x

--- What-If: Reallocate to 10K vocab ---
  10K vocab embedding: 20,480,000 params
  Freed params:        110,592,000
  New transformer capacity: 1,079,566,336 (1.1x current)


## Visualization

In [5]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

if not results:
    print("No models loaded. Check errors above.")
else:
    # Add gated model estimates to results for plotting
    all_results = []

    # Gated estimates
    for name, vocab, hidden, layers, tied in gated_models:
        emb = vocab * hidden
        trans_est = layers * 12 * hidden * hidden
        total_est = emb + trans_est + (0 if tied else emb)
        vocab_cost = emb if tied else emb * 2
        all_results.append({
            "model": name,
            "total": total_est,
            "vocab": vocab_cost,
            "transformer": trans_est,
            "vocab_pct": vocab_cost / total_est * 100,
            "transformer_pct": trans_est / total_est * 100,
            "vocab_size": vocab,
        })

    # Real audits
    for r in results:
        r_copy = r.copy()
        r_copy["model"] = r["model"].split("/")[-1]
        all_results.append(r_copy)

    # Sort by total params
    all_results.sort(key=lambda x: x["total"])

    # --- Plot 1: Stacked bar chart ---
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    names = [r["model"] for r in all_results]
    vocab_pcts = [r["vocab_pct"] for r in all_results]
    trans_pcts = [r["transformer_pct"] for r in all_results]
    other_pcts = [100 - v - t for v, t in zip(vocab_pcts, trans_pcts)]

    x = range(len(names))
    bars1 = axes[0].bar(x, vocab_pcts, color="#e74c3c", label="Vocabulary (embeddings)")
    bars2 = axes[0].bar(x, trans_pcts, bottom=vocab_pcts, color="#2ecc71", label="Transformer (reasoning)")
    bars3 = axes[0].bar(x, other_pcts,
                        bottom=[v + t for v, t in zip(vocab_pcts, trans_pcts)],
                        color="#95a5a6", label="Other")

    axes[0].set_xticks(x)
    axes[0].set_xticklabels(names, rotation=35, ha="right", fontsize=9)
    axes[0].set_ylabel("% of Total Parameters")
    axes[0].set_title("Parameter Allocation: Vocabulary vs Reasoning")
    axes[0].legend(loc="upper right", fontsize=8)
    axes[0].set_ylim(0, 105)

    # Add % labels on vocab bars
    for bar, pct in zip(bars1, vocab_pcts):
        if pct > 5:
            axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
                        f"{pct:.0f}%", ha="center", va="center",
                        fontweight="bold", color="white", fontsize=9)

    # --- Plot 2: Vocab size vs vocab overhead % ---
    vocab_sizes = [r["vocab_size"] for r in all_results]
    totals_B = [r["total"] / 1e9 for r in all_results]

    scatter = axes[1].scatter(vocab_sizes, vocab_pcts, s=[t * 300 for t in totals_B],
                             c=vocab_pcts, cmap="RdYlGn_r", edgecolors="black",
                             alpha=0.8, zorder=5)
    for i, name in enumerate(names):
        axes[1].annotate(name, (vocab_sizes[i], vocab_pcts[i]),
                        textcoords="offset points", xytext=(8, 5), fontsize=8)

    axes[1].set_xlabel("Vocabulary Size (tokens)")
    axes[1].set_ylabel("Vocabulary Overhead (% of params)")
    axes[1].set_title("Larger Vocab = More Wasted Parameters")
    axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}K"))
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("paatra_step1_allocation.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\nSaved: paatra_step1_allocation.png")


Saved: paatra_step1_allocation.png


## Summary Table

In [6]:
if results or gated_models:
    print(f"{'Model':<28} {'Total':>10} {'Vocab %':>9} {'Trans %':>9} {'Vocab Size':>11}")
    print("─" * 70)
    for r in all_results:
        total_str = f"{r['total']/1e6:.0f}M"
        print(f"{r['model']:<28} {total_str:>10} {r['vocab_pct']:>8.1f}% {r['transformer_pct']:>8.1f}% {r['vocab_size']:>10,}")
    print()
    print("Key insight: The smaller the model + the larger the inherited vocab = the worse the allocation.")
    print("Gemma 3 270M is the extreme case: 44% vocabulary, 56% reasoning.")
    print("SmolLM2 (49K vocab) and TinyLlama (32K vocab) show that smaller vocabs fix this.")

Model                             Total   Vocab %   Trans %  Vocab Size
──────────────────────────────────────────────────────────────────────
SmolLM2-360M                       362M     13.0%     87.0%     49,152
Qwen2.5-0.5B                       494M     27.6%     72.4%    151,936
Gemma 3 270M                       912M     44.1%     55.9%    262,144
TinyLlama-1.1B-Chat-v1.0          1100M     11.9%     88.1%     32,000
Llama 3.2 1B                      1331M     39.5%     60.5%    128,256
Gemma 2 2B                        2246M     26.3%     73.7%    256,000
Llama 3.2 3B                      3959M     19.9%     80.1%    128,256

Key insight: The smaller the model + the larger the inherited vocab = the worse the allocation.
Gemma 3 270M is the extreme case: 44% vocabulary, 56% reasoning.
SmolLM2 (49K vocab) and TinyLlama (32K vocab) show that smaller vocabs fix this.


---

## Optional: Audit Gated Models (requires HuggingFace login)

If you have access to Gemma or Llama on HuggingFace, uncomment and run below.

In [7]:
# Uncomment these lines if you have HuggingFace access:

# from huggingface_hub import login
# login()  # paste your token when prompted

# gated_results = []
# for m in ["google/gemma-3-270m-it", "meta-llama/Llama-3.2-1B"]:
#     gated_results.append(audit_model(m))